# LeetCode #1421: Finding the Users Active Minutes

https://leetcode.com/problems/finding-the-users-active-minutes/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^2)$ | $O(n)$ |
| **Optimal: Hash Map of Sets ★** | $O(n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
For each unique user, scan all logs to collect their distinct active minutes, then count them. This double-loop is $O(n^2)$ because each user is counted by iterating the full log.

### Optimal: Hash Map of Sets ★
One pass: for each log entry, insert the minute into the set for that user ID. Duplicate minutes are deduplicated by the set automatically. A second linear pass counts the set size for each user and increments the answer bucket. Total work is $O(n)$ with $O(n)$ space.

**Constraints:**
* 1 <= logs.length <= 10^4
* 0 <= user_id <= 10^5
* 1 <= time <= 10^5
* k >= logs.length

## Solutions
### C#

In [ ]:
// Hash Map of Sets: one pass to deduplicate per-user minutes, one pass to count UAM
public class Solution {
    public int[] FindingUsersActiveMinutes(int[][] logs, int k) {
        // Map each user to their set of unique active minutes
        var userMinutes = new Dictionary<int, HashSet<int>>();
        foreach (var log in logs) {
            int user = log[0], time = log[1];
            if (!userMinutes.ContainsKey(user))
                userMinutes[user] = new HashSet<int>();
            // Sets ignore duplicates — UAM is just the set size
            userMinutes[user].Add(time);
        }

        // Build answer: answer[j] = count of users with exactly j+1 unique minutes
        int[] answer = new int[k];
        foreach (var kv in userMinutes) {
            int uam = kv.Value.Count;
            answer[uam - 1]++;
        }
        return answer;
    }
}

### Python

In [ ]:
# Hash Map of Sets: one pass to deduplicate per-user minutes, one pass to count UAM
from typing import List
from collections import defaultdict

class Solution:
    def findingUsersActiveMinutes(self, logs: List[List[int]], k: int) -> List[int]:
        # Map each user to their set of unique active minutes
        user_minutes = defaultdict(set)
        for user, time in logs:
            # Adding to a set automatically deduplicates same-minute actions
            user_minutes[user].add(time)

        # Build answer array: index j holds count of users with exactly j+1 UAM
        answer = [0] * k
        for minutes in user_minutes.values():
            uam = len(minutes)
            answer[uam - 1] += 1
        return answer

### Go

In [ ]:
// Hash Map of Sets: one pass to deduplicate per-user minutes, one pass to count UAM
package main

func findingUsersActiveMinutes(logs [][]int, k int) []int {
    // Map each user ID to a set of unique minutes (represented as a map to struct{})
    userMinutes := make(map[int]map[int]struct{})
    for _, log := range logs {
        user, time := log[0], log[1]
        if _, exists := userMinutes[user]; !exists {
            userMinutes[user] = make(map[int]struct{})
        }
        // Empty struct value: zero-size map entry acts as a set element
        userMinutes[user][time] = struct{}{}
    }

    // Count how many users have each UAM value
    answer := make([]int, k)
    for _, minutes := range userMinutes {
        uam := len(minutes)
        answer[uam-1]++
    }
    return answer
}

### Rust

In [ ]:
// Hash Map of Sets: one pass to deduplicate per-user minutes, one pass to count UAM
use std::collections::{HashMap, HashSet};

impl Solution {
    pub fn finding_users_active_minutes(logs: Vec<Vec<i32>>, k: i32) -> Vec<i32> {
        // Map each user to their set of unique active minutes
        let mut user_minutes: HashMap<i32, HashSet<i32>> = HashMap::new();
        for log in &logs {
            let (user, time) = (log[0], log[1]);
            // HashSet deduplicates repeated minute values automatically
            user_minutes.entry(user).or_default().insert(time);
        }

        // Build answer: index j = count of users with exactly j+1 distinct minutes
        let mut answer = vec![0i32; k as usize];
        for minutes in user_minutes.values() {
            let uam = minutes.len();
            answer[uam - 1] += 1;
        }
        answer
    }
}

## Example Scenarios

**1. Common Case** — Two users, some shared minutes

**Input:** `logs = [[0,5],[1,2],[0,2],[0,5],[1,3]], k = 5`
User 0 has unique minutes \{5, 2\} → UAM = 2. User 1 has unique minutes \{2, 3\} → UAM = 2. Answer: `[0,2,0,0,0]` — two users each with UAM = 2.

**2. Slightly Complex** — One user with many duplicate entries

**Input:** `logs = [[1,1],[1,2],[1,3],[1,1],[1,2]], k = 5`
User 1's minutes \{1, 2, 3\} deduplicate to 3 unique values. The set discards the two repeated entries silently. Answer: `[0,0,1,0,0]`.

**3. Edge Case: Time Factor** — Single log entry

**Input:** `logs = [[0,1]], k = 1`
Only one log entry means one user with UAM = 1. The hash map has one entry and the pass is $O(1)$. Answer: `[1]`.

**4. Edge Case: Space Factor** — All users distinct, all minutes distinct

**Input:** `logs = [[i, i] for i in range(n)], k = 1` (n = 10,000)
Every user appears exactly once with one unique minute, so UAM = 1 for all. The hash map stores 10,000 singleton sets, the maximum $O(n)$ space. Answer: `[10000, 0, ...]`.

**5. Almost-Impossible but Plausible** — All actions in one minute for one user

**Input:** `logs = [[0, 99999]] * 10000, k = 2`
All 10,000 entries belong to user 0 at minute 99,999. The set reduces to size 1, so UAM = 1. Despite 10,000 log entries, the answer is `[1, 0]` — only $O(1)$ unique state is retained.